In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import pycountry_convert as pc



In [2]:
GDP_PATH = "/Users/tonytony/Final Project/Data/Raw/gdp.csv"

print("GDP path:", GDP_PATH)


GDP path: /Users/tonytony/Final Project/Data/Raw/gdp.csv


In [3]:
gdp_df = pd.read_csv(GDP_PATH, skiprows=4)

print("GDP dataset loaded successfully")
print("Shape:", gdp_df.shape)

gdp_df.head()


GDP dataset loaded successfully
Shape: (266, 70)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,27441.550214,28440.041688,30082.158423,30645.890602,22759.807175,26749.329609,30975.998912,35718.753119,39498.594129,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089204,186.909053,197.367547,225.400079,208.962717,226.836135,...,1334.470500,1528.104224,1552.073722,1507.085600,1351.591669,1562.416175,1679.327622,1571.449189,1615.396356,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,522.082216,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,NaN,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,121.936832,127.451040,133.823783,139.004980,148.545883,155.561897,...,1630.039439,1574.230564,1720.140092,2216.385055,2030.861659,2112.794076,2138.473153,1841.855064,1411.337029,NaN
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2051.814621,2790.718869,2860.093648,2493.678844,1759.356199,2303.908127,3682.113151,2916.136633,2665.874448,NaN


In [4]:
gdp_df = gdp_df.drop(
    columns=[col for col in gdp_df.columns if str(col).startswith("Unnamed")],
    errors="ignore"
)

year_cols = [col for col in gdp_df.columns if str(col).isdigit()]
gdp_df[year_cols] = gdp_df[year_cols].apply(pd.to_numeric, errors="coerce")

print("Number of year columns:", len(year_cols))
print("Year range:", year_cols[0], "-", year_cols[-1])
print("Shape after cleaning:", gdp_df.shape)

gdp_df.head()


Number of year columns: 65
Year range: 1960 - 2024
Shape after cleaning: (266, 69)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,27458.220154,27441.550214,28440.041688,30082.158423,30645.890602,22759.807175,26749.329609,30975.998912,35718.753119,39498.594129
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089204,186.909053,197.367547,225.400079,208.962717,226.836135,...,1498.875240,1334.470500,1528.104224,1552.073722,1507.085600,1351.591669,1562.416175,1679.327622,1571.449189,1615.396356
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,565.569730,522.082216,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,121.936832,127.451040,133.823783,139.004980,148.545883,155.561897,...,1860.727686,1630.039439,1574.230564,1720.140092,2216.385055,2030.861659,2112.794076,2138.473153,1841.855064,1411.337029
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,3641.728939,2051.814621,2790.718869,2860.093648,2493.678844,1759.356199,2303.908127,3682.113151,2916.136633,2665.874448


In [5]:
SPECIAL_COUNTRY_CODES = {"XKX", "SXM"}

def is_country_code(code):
    code = str(code).strip()
    if code in SPECIAL_COUNTRY_CODES:
        return True
    try:
        alpha2 = pc.country_alpha3_to_country_alpha2(code)
        pc.country_alpha2_to_continent_code(alpha2)
        return True
    except:
        return False

gdp_df["Is Country"] = gdp_df["Country Code"].astype(str).apply(is_country_code)

print("Country / territory rows:", int(gdp_df["Is Country"].sum()))
print("Aggregate rows:", int((~gdp_df["Is Country"]).sum()))

gdp_df[["Country Name", "Country Code", "Is Country"]].head()


Country / territory rows: 215
Aggregate rows: 51


,Country Name,Country Code,Is Country
0,Aruba,ABW,True
1,Africa Eastern and Southern,AFE,False
2,Afghanistan,AFG,True
3,Africa Western and Central,AFW,False
4,Angola,AGO,True


In [6]:
gdp_geo = gdp_df.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code", "Is Country"],
    value_vars=year_cols,
    var_name="Year",
    value_name="GDP per capita (current US$)"
)

gdp_geo["Year"] = pd.to_numeric(gdp_geo["Year"], errors="coerce").astype("Int64")
gdp_geo["GDP per capita (current US$)"] = pd.to_numeric(
    gdp_geo["GDP per capita (current US$)"],
    errors="coerce"
)

print("Long format shape:", gdp_geo.shape)

gdp_geo.head()


Long format shape: (17290, 7)


,Country Name,Country Code,Indicator Name,Indicator Code,Is Country,Year,GDP per capita (current US$)
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,True,1960,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,False,1960,186.089204
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,True,1960,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,False,1960,121.936832
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,True,1960,NaN


In [13]:
gdp_geo_clean = gdp_geo[
    (gdp_geo["Is Country"]) &
    (gdp_geo["GDP per capita (current US$)"].notna()) &
    (gdp_geo["GDP per capita (current US$)"] > 0)
].copy()

gdp_geo_clean["Year Label"] = gdp_geo_clean["Year"].astype(str)
gdp_geo_clean["log10 GDP per capita"] = np.log10(gdp_geo_clean["GDP per capita (current US$)"])

print("Clean geo data shape:", gdp_geo_clean.shape)
print("Year range:", int(gdp_geo_clean["Year"].min()), "-", int(gdp_geo_clean["Year"].max()))
print("Number of countries/territories:", gdp_geo_clean["Country Code"].nunique())

gdp_geo_clean.head()


Clean geo data shape: (11509, 9)
Year range: 1960 - 2024
Number of countries/territories: 212


,Country Name,Country Code,Indicator Name,Indicator Code,Is Country,Year,GDP per capita (current US$),Year Label,log10 GDP per capita
9,Argentina,ARG,GDP per capita (current US$),NY.GDP.PCAP.CD,True,1960,778.251707,1960,2.891120
13,Australia,AUS,GDP per capita (current US$),NY.GDP.PCAP.CD,True,1960,1813.431099,1960,3.258501
14,Austria,AUT,GDP per capita (current US$),NY.GDP.PCAP.CD,True,1960,939.914815,1960,2.973088
16,Burundi,BDI,GDP per capita (current US$),NY.GDP.PCAP.CD,True,1960,70.905100,1960,1.850677
17,Belgium,BEL,GDP per capita (current US$),NY.GDP.PCAP.CD,True,1960,1290.286072,1960,3.110686


In [8]:
gdp_coverage = (
    gdp_geo_clean
    .groupby("Year")
    .agg(
        country_count=("Country Code", "nunique"),
        min_gdp=("GDP per capita (current US$)", "min"),
        median_gdp=("GDP per capita (current US$)", "median"),
        mean_gdp=("GDP per capita (current US$)", "mean"),
        max_gdp=("GDP per capita (current US$)", "max")
    )
    .reset_index()
)

print("GDP coverage by year:")
gdp_coverage.tail(15)


GDP coverage by year:


,Year,country_count,min_gdp,median_gdp,mean_gdp,max_gdp
50,2010,211,216.727705,5843.533768,16271.174111,161853.920622
51,2011,212,230.069761,6527.674656,17918.205501,179363.984222
52,2012,210,231.098788,6574.009584,17797.544040,165444.651379
53,2013,210,234.844770,7056.442746,18424.932773,184940.663805
54,2014,211,250.544492,7146.704591,18731.015970,195693.570739
55,2015,210,254.402585,6147.913295,16902.776277,170437.101188
56,2016,209,232.937812,5982.370301,17084.850351,173604.753556
57,2017,209,246.060726,6548.036538,17960.284445,170663.375248
58,2018,209,245.661678,6978.491215,19023.219652,188298.315668
59,2019,209,234.310616,7037.008037,18960.205880,193746.785647


In [21]:
year_order = [str(year) for year in sorted(gdp_geo_clean["Year"].dropna().astype(int).unique())]

color_min = gdp_geo_clean["GDP per capita (current US$)"].quantile(0.02)
color_max = gdp_geo_clean["GDP per capita (current US$)"].quantile(0.98)

gdp_animation = px.choropleth(
    gdp_geo_clean,
    locations="Country Code",
    locationmode="ISO-3",
    color="GDP per capita (current US$)",
    animation_frame="Year Label",
    animation_group="Country Code",
    hover_name="Country Name",
    hover_data={
        "Country Code": False,
        "Year": True,
        "GDP per capita (current US$)": ":,.2f"
    },
    category_orders={"Year Label": year_order},
    color_continuous_scale="YlGnBu",
    range_color=[color_min, color_max],
    projection="natural earth",
    title="GDP per capita by Year",
    labels={
        "GDP per capita (current US$)": "GDP per capita (current US$)"
    }
)

gdp_animation.update_layout(
    height=650,
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(title="GDP per capita")
)

gdp_animation.show()


- Countries in North America, Western Europe, parts of East Asia, and Oceania generally have higher GDP per capita than many other regions.
- Many countries in Sub-Saharan Africa remain at lower GDP per capita levels throughout most of the observed period.
- Over time, GDP per capita generally increases in many countries, especially after the 2000s.
- However, growth is not evenly distributed across regions, indicating that economic gaps between groups of countries remain large.
- Some countries have extremely high GDP per capita compared to the rest, creating outliers in the distribution.

**Conclusion:** GDP per capita is highly uneven across geographic regions and levels of economic development. Therefore, median values, rankings, or log scale can be useful to reduce the influence of extreme outliers.